# Prática 27/10 · O problema: onde o eleitor de Mato Grosso deixa de votar?
**Aprendizagem Computacional · 4º TIA · Aula 01 · versão INICIAL**

Esta é a primeira prática do **PPI-molde Comparece MT**, o projeto de exemplo que vamos construir, um pedaço por semana, até a banca. Cada pedaço corresponde a uma entrega da PPI de vocês.

Hoje não tem modelo. Tem o passo que vem antes de todos: **entender o problema e escrever a pergunta**, que é o que o grupo entrega no check-in zero (03/11).

**O problema público:** no 1º turno de 2026, mais de 1 em cada 5 eleitores aptos de MT (22%) não votou. Um comitê de mobilização (TRE-MT e prefeituras) tem orçamento para ações de transporte e comunicação no dia da eleição. **Onde investir?**

Para responder antes da próxima eleição, precisamos prever a abstenção de cada município com o que se sabe **antes** dela. Hoje exploramos a base que o ETL da semana que vem vai construir, e preenchemos a ficha do problema.

| TODO | Tarefa | Apostila |
|---|---|---|
| 1 | Recalcular y a partir das contagens | cap. 2 e 3 |
| 2 | Média simples × média ponderada | cap. 4 |
| 3 | Os municípios com maior abstenção | cap. 4 |
| 4 | O mapa da abstenção | cap. 4 |
| 5 | 2022 × 2026: a abstenção se repete? | cap. 4 |
| 6 | A ficha do problema | cap. 1 a 3 |

## 1 · A base (pronto)
Uma linha por município de MT. Nesta prática, a base já vem pronta; em 03/11 vocês a constroem a partir do TSE.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
pd.set_option('display.precision', 2)

ARQUIVO = 'base_analitica.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-10-27_Aula01_O_Problema/dados/' + ARQUIVO
candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
base = pd.read_csv(local if local is not None else URL, dtype={'CD_MUNICIPIO': str})
print('Base lida de:', local.resolve() if local is not None else URL, '|', base.shape)

In [ ]:
base.head()

## 2 · TODO 1: de onde vem o y? (cap. 2 e 3)
O alvo é `ABSTENCAO_PCT`: a porcentagem dos eleitores **aptos** que **não votaram** no 1º turno para presidente em 2026.
Recalcule a partir das contagens: `100 * base['QT_ABSTENCOES'] / base['QT_APTOS']`, e confira que bate com a coluna.

Saber de onde vem o y é a primeira pergunta da banca, e evita um erro comum: usar em X uma coluna com a qual se calcula y.

In [ ]:
y_recalculado = None   # TODO 1: 100 × abstenções ÷ aptos
print('bate com ABSTENCAO_PCT?', np.allclose(y_recalculado, base['ABSTENCAO_PCT']))

## 3 · TODO 2: a abstenção "de MT" é qual? (cap. 4)
Há duas médias possíveis:
- **média simples** dos municípios: cada município pesa 1, seja Cuiabá ou Araguainha;
- **média ponderada**, a do estado: total de abstenções ÷ total de aptos.

Calcule as duas. A diferença mostra por que a **unidade de análise** importa: o nosso modelo vai olhar municípios, não eleitores.

In [ ]:
media_simples = None   # TODO 2: média da coluna ABSTENCAO_PCT
media_estado = None    # TODO 2: 100 × soma das abstenções ÷ soma dos aptos
print(f'média simples dos municípios: {media_simples:.2f}% | abstenção do estado: {media_estado:.2f}%')
print('municípios com abstenção de 25% ou mais:', (base['ABSTENCAO_PCT'] >= 25).sum(), 'de', len(base))

**Pergunta para discussão:** por que a abstenção do estado é **menor** que a média simples dos municípios?

**Resposta esperada:** porque os municípios grandes (Cuiabá, Várzea Grande, Rondonópolis) têm abstenção abaixo da média e pesam muito no total de eleitores. Na média simples, eles valem o mesmo que um município de mil eleitores. Para o comitê, as duas importam: a ponderada diz quantas **pessoas** faltaram; a simples diz como os **municípios** se distribuem.

## 4 · TODO 3: onde a abstenção é maior? (cap. 4)
Mostre os 10 municípios com maior abstenção em 2026, com `base.nlargest(10, 'ABSTENCAO_PCT')` e as colunas `NM_MUNICIPIO`, `ABSTENCAO_PCT`, `QT_APTOS` e `DIST_CUIABA_KM`.

In [ ]:
top10 = None   # TODO 3: os 10 maiores em ABSTENCAO_PCT, com as 4 colunas pedidas
top10

## 5 · TODO 4: o mapa (cap. 4)
Cada ponto é a sede de um município, na sua latitude e longitude. Complete o `scatter` colorindo pela abstenção: `c=base['ABSTENCAO_PCT']`. A estrela é Cuiabá.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
pontos = ax.scatter(base['LONGITUDE'], base['LATITUDE'], c=None,   # TODO 4: colorir pela abstenção
                    cmap='YlOrRd', s=10 + base['QT_APTOS'] / 2000, edgecolor='grey', linewidth=0.3)
ax.scatter([-56.097], [-15.601], marker='*', s=250, color='#1f6feb', label='Cuiabá')
plt.colorbar(pontos, label='Abstenção 2026 (%)'); ax.legend(loc='lower left')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude'); ax.set_title('Abstenção por município (tamanho = eleitores aptos)')
plt.tight_layout(); plt.show()
print('correlação entre distância até Cuiabá e abstenção:', round(base['DIST_CUIABA_KM'].corr(base['ABSTENCAO_PCT']), 2))

**Pergunta para discussão:** o mapa sugere que longe da capital se vota menos. Isso é uma **causa**?

**Resposta esperada:** ainda não sabemos. Distância pode significar seção eleitoral longe de casa, estrada ruim, população que migrou e não transferiu o título. O mapa levanta a hipótese; o projeto testa e o artigo discute. Correlação não é causalidade.

## 6 · TODO 5: a abstenção se repete? (cap. 4)
Se um município faltou muito em 2022, falta muito em 2026? Calcule a correlação entre `ABST_2022_PCT` e `ABSTENCAO_PCT` com `.corr()`.
A célula também conta em quantos municípios a abstenção caiu de uma eleição para a outra.

In [ ]:
r_2022 = None   # TODO 5: correlação entre ABST_2022_PCT e ABSTENCAO_PCT
caiu = (base['ABSTENCAO_PCT'] < base['ABST_2022_PCT']).sum()
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(base['ABST_2022_PCT'], base['ABSTENCAO_PCT'], s=14, alpha=0.6)
ax.plot([15, 36], [15, 36], '--', color='grey', label='igual a 2022')
ax.set_xlabel('Abstenção 2022 (%)'); ax.set_ylabel('Abstenção 2026 (%)'); ax.legend(); plt.tight_layout(); plt.show()
print(f'correlação 2022 × 2026: {r_2022:.2f} | a abstenção caiu em {caiu} de {len(base)} municípios')

Dois fatos para guardar: a abstenção **se repete** (correlação alta), e ela **caiu** em quase todo o estado de 2022 para 2026. Por isso, mais adiante, a régua "repetir 2022" vai ser corrigida pela queda média.

## 7 · TODO 6: a ficha do problema (cap. 1 a 3)
É a entrega do check-in zero. Preencha o dicionário com uma frase por item, para o PPI-molde. Regras:
- `y` é o **nome da coluna** do alvo;
- `X` é uma **lista de colunas** da base, só com o que se sabe **antes** da eleição (nada de contagens de 2026);
- `unidade` diz o que é uma linha.

In [ ]:
ficha = {
    'problema': 'Mais de 1 em cada 5 eleitores aptos de MT (22%) não votou no 1º turno de 2026.',
    'decisor': '',    # TODO 6: quem decide?
    'decisao': '',    # TODO 6: o que essa pessoa decide com o resultado?
    'pergunta': '',   # TODO 6: a pergunta analítica, em uma frase
    'unidade': '',    # TODO 6: o que é uma linha da base?
    'y': '',          # TODO 6: o nome da coluna do alvo
    'X': [],          # TODO 6: lista de colunas conhecidas ANTES da eleição
}
for item, valor in ficha.items():
    print(f'{item:>9}: {valor}')

## Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None ou ...) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('y recalculado bate com a coluna', lambda: np.allclose(y_recalculado, base['ABSTENCAO_PCT'])),
    conferir('média simples 23,45% e do estado 22,20%', lambda: round(media_simples, 2) == 23.45 and round(media_estado, 2) == 22.2),
    conferir('maior abstenção: Nova Nazaré', lambda: top10.iloc[0]['NM_MUNICIPIO'] == 'NOVA NAZARÉ' and len(top10) == 10),
    conferir('correlação 2022 × 2026 = 0,77', lambda: round(r_2022, 2) == 0.77),
    conferir('ficha: y é a coluna ABSTENCAO_PCT', lambda: ficha['y'] == 'ABSTENCAO_PCT'),
    conferir('ficha: X só com colunas da base, sem contagens de 2026 nem y', lambda: len(ficha['X']) > 0 and set(ficha['X']) <= set(base.columns) and not {'QT_APTOS', 'QT_ABSTENCOES', 'ABSTENCAO_PCT'} & set(ficha['X'])),
    conferir('ficha: decisor, decisão, pergunta e unidade preenchidos', lambda: all(len(ficha[k]) > 10 for k in ['decisor', 'decisao', 'pergunta', 'unidade'])),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist do check-in zero (03/11)
- [ ] Pergunta analítica em uma frase, com a decisão e quem decide.
- [ ] Unidade de análise definida (o que é uma linha?).
- [ ] y e X candidatos escritos, e só com o que se sabe antes do fato.
- [ ] Pelo menos uma fonte pública baixada, com link e data do download.